In [ ]:
emb_matrix = np.vstack(products["embedding"].to_numpy())
short_emb_matrix = np.vstack(products["embedding_short"].to_numpy())
short_emb_matrix2 = np.vstack(products["embedding_short2"].to_numpy())


idx = 6  # any product row

sims = cosine_similarity([emb_matrix[idx]], emb_matrix)[0]
sims.shape
    
top_idx = sims.argsort()[::-1][1:11]  # skip itself

print("Query product:", products.iloc[idx]["Description"])


for i in top_idx:
    print(products.iloc[i]["StockCode"], "-", products.iloc[i]["Description"], "-", round(sims[i], 3))


sims = cosine_similarity([short_emb_matrix[idx]], short_emb_matrix)[0]

top_idx = sims.argsort()[::-1][1:11]

print("Query product:", products.iloc[idx]["Description"])
print()

for i in top_idx:
    print(
        products.iloc[i]["StockCode"],
        "-",
        products.iloc[i]["Description"],
        "-",
        round(sims[i], 3)
    )


sims = cosine_similarity([short_emb_matrix2[idx]], short_emb_matrix2)[0]

top_idx = sims.argsort()[::-1][1:11]

print("Query product:", products.iloc[idx]["Description"])
print()
`
for i in top_idx:
    print(
        products.iloc[i]["StockCode"],
        "-",
        products.iloc[i]["Description"],
        "-",
        round(sims[i], 3)
    )

In [ ]:
embeddings_short = model.encode(
    products["short_description"].tolist(),
    show_progress_bar=True
)

embeddings_short2 = model.encode(
    products["short_description2"].tolist(),
    show_progress_bar=True
)


products["embedding_short"] = list(embeddings_short)
products["embedding_short2"] = list(embeddings_short2)

In [ ]:

# Customer-level totals
customer_price_stats = (
    df.groupby('Customer ID', as_index=False)
      .agg(
          total_quantity=('Quantity', 'sum'),
          total_line_amount=('line_amount', 'sum')
      )
)

# Average item price per customer
customer_price_stats['avg_item_price'] = (
    customer_price_stats['total_line_amount']
    / customer_price_stats['total_quantity']
)

# Customers to keep:
# avg_item_price > 0 and <= 1000
valid_customers = customer_price_stats.loc[
    (customer_price_stats['avg_item_price'] > 0) &
    (customer_price_stats['avg_item_price'] <= 100),
    'Customer ID'
]

# Filter the original dataframe
df = df[df['Customer ID'].isin(valid_customers)].copy()

print("Remaining rows:", len(df))
print("Remaining customers:", df['Customer ID'].nunique())

removed_customers = customer_price_stats.loc[
    (customer_price_stats['avg_item_price'] <= 0) |
    (customer_price_stats['avg_item_price'] > 100)
]

print("Customers removed:", len(removed_customers))
